# EN3150 Assignment 03 — Part 3: Optimizer selection and tuning

**Task:** state the optimizer and learning rate you chose and why; compare it with (a) plain SGD and (b) SGD with momentum; explain the effect of the momentum parameter on convergence and performance.

**Experiment:** Model A is trained from the *same initial weights* (fixed seed) and the same data for 20 epochs with each optimizer. Selection is by **validation** accuracy; test accuracy is only reported.

| Run | Optimizer | Learning rate |
|---|---|---|
| Chosen | Adam | 1e-3 |
| (a) | SGD | 0.01 |
| (b) | SGD + momentum 0.9 | 0.01 |
| Extra | Momentum sweep: 0, 0.5, 0.9, 0.99 (+ 0.99 with a smaller lr) | 0.01 |

Estimated time on a CPU laptop: roughly 20 epochs x (3 main runs + 3 extra sweep runs) x Model A epoch time. Set `RUN_SWEEP = False` to skip the sweep while testing.

**Before running:** keep `a03_common.py` in the same folder as this notebook, and run the **Model A notebook first** (it creates `outputs/eurosat_64.npz` and `outputs/splits.npz`, so every model uses the identical 70/15/15 split).

## 1. Setup

In [ ]:
from a03_common import *

EPOCHS_OPT = 20
RUN_SWEEP = True
set_seed(SEED)
data = Data(norm="dataset", batch_size=64, augment=True)

def run(name, make_opt, epochs=EPOCHS_OPT, verbose=True):
    set_seed(SEED)                                   # identical initial weights and batch order for every run
    model = ModelA(data.num_classes, 3, IMG_SIZE).to(DEVICE)
    hist = train_model(model, make_opt(model.parameters()), data, epochs, name=name, verbose=verbose)
    ev = evaluate(model, data, verbose=False)
    return {"hist": hist, "test_acc": ev["acc"]}

def epochs_to(hist, thr):
    for i, a in enumerate(hist["val_acc"], 1):
        if a >= thr:
            return i
    return None

def table(results, thr=0.90):
    print(f"{'Run':<34}{'Best val acc':>13}{'Ep to '+str(int(thr*100))+'%':>10}{'Val loss @5':>13}{'Final train loss':>18}{'Test acc':>10}{'s/epoch':>9}")
    for name, r in results.items():
        h = r["hist"]
        e = epochs_to(h, thr)
        print(f"{name:<34}{max(h['val_acc']):>13.4f}{(str(e) if e else 'not hit'):>10}{h['val_loss'][4]:>13.4f}"
              f"{h['train_loss'][-1]:>18.4f}{r['test_acc']:>10.4f}{np.mean(h['epoch_time']):>9.1f}")

## 2. Main comparison: Adam vs SGD vs SGD + momentum
**Why Adam, lr = 1e-3 (the chosen optimizer):**
- Adam keeps running estimates of the gradient mean and squared gradient, giving every parameter its own adaptive step size. It needs little learning-rate tuning, which matters when each experiment costs minutes of CPU time.
- Model A has no normalisation layers, so layers see differently scaled gradients. Adaptive scaling copes with that better than a single global learning rate.
- 1e-3 is the well-tested default for Adam and converges within the 20-25 epoch budget.
- The extra optimizer state (two tensors per weight) exists only during training, so it adds nothing to the deployed model size or inference cost.

The comparison runs below test whether that reasoning holds on this dataset.

In [ ]:
RUNS = {
    "Adam (lr=1e-3)":                lambda p: torch.optim.Adam(p, lr=1e-3),
    "SGD (lr=0.01)":                 lambda p: torch.optim.SGD(p, lr=0.01),
    "SGD + momentum 0.9 (lr=0.01)":  lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
}
results = {}
for name, make_opt in RUNS.items():
    results[name] = run(name, make_opt)

In [ ]:
table(results)
hists = {k: v["hist"] for k, v in results.items()}
plot_compare(hists, "val_loss",  "Validation loss",     "Validation loss: optimizer comparison",     "opt_val_loss.png")
plot_compare(hists, "val_acc",   "Validation accuracy", "Validation accuracy: optimizer comparison", "opt_val_acc.png")
plot_compare(hists, "train_loss","Training loss",       "Training loss: optimizer comparison",       "opt_train_loss.png")

## 3. Impact of the momentum parameter
PyTorch's SGD with momentum $\beta$ and learning rate $\eta$:

$$v_{t+1} = \beta\, v_t + \nabla L(w_t), \qquad w_{t+1} = w_t - \eta\, v_{t+1}$$

- **Speed-up:** along a direction where the gradient keeps the same sign, $v$ accumulates to about $g/(1-\beta)$, so the *effective* step is $\eta/(1-\beta)$: about **10x** larger for $\beta = 0.9$ and **100x** for $\beta = 0.99$.
- **Damping:** along directions where the gradient flips sign every step (zig-zag across a narrow valley), the contributions cancel inside $v$, so oscillation shrinks while progress along the valley floor continues.
- **Too small** ($\beta \to 0$): behaves like plain SGD, slow on flat or ill-conditioned regions.
- **Too large** ($\beta = 0.99$): the velocity has a long memory, so the optimizer overshoots minima, oscillates, or diverges unless the learning rate is reduced; it also reacts slowly when the gradient direction changes.
- **Interaction with $\eta$:** only the product $\eta/(1-\beta)$ matters for step size, so a larger $\beta$ should be paired with a smaller $\eta$. The extra run `beta=0.99, lr=0.001` tests this.

In [ ]:
MOMENTA = [0.0, 0.5, 0.9, 0.99]
sweep = {}
if RUN_SWEEP:
    for beta in MOMENTA:
        key = f"momentum {beta} (lr=0.01)"
        if beta == 0.0:
            sweep[key] = results["SGD (lr=0.01)"]
        elif beta == 0.9:
            sweep[key] = results["SGD + momentum 0.9 (lr=0.01)"]
        else:
            sweep[key] = run(key, lambda p, b=beta: torch.optim.SGD(p, lr=0.01, momentum=b), verbose=False)
    key = "momentum 0.99 (lr=0.001)"
    sweep[key] = run(key, lambda p: torch.optim.SGD(p, lr=0.001, momentum=0.99), verbose=False)

    table(sweep)
    sh = {k: v["hist"] for k, v in sweep.items()}
    plot_compare(sh, "val_loss", "Validation loss",     "Momentum sweep: validation loss",     "momentum_val_loss.png")
    plot_compare(sh, "val_acc",  "Validation accuracy", "Momentum sweep: validation accuracy", "momentum_val_acc.png")
else:
    print("Sweep skipped (RUN_SWEEP = False).")

**Toy illustration (no training needed).** Minimising the ill-conditioned bowl $f(w) = \tfrac12(w_1^2 + 25\,w_2^2)$, the same shape as a narrow valley in the loss surface. Plain gradient descent zig-zags across the steep direction and crawls along the shallow one; momentum damps the zig-zag and speeds up the crawl. This is a good small figure for the report.

In [ ]:
def descend(beta, lr=0.03, steps=60):
    w, v = np.array([4.0, 1.0]), np.zeros(2)
    path = [w.copy()]
    for _ in range(steps):
        g = np.array([w[0], 25 * w[1]])
        v = beta * v + g
        w = w - lr * v
        path.append(w.copy())
    return np.array(path)

xs, ys = np.meshgrid(np.linspace(-1, 4.5, 200), np.linspace(-1.2, 1.5, 200))
zs = 0.5 * (xs ** 2 + 25 * ys ** 2)
plt.figure(figsize=(8, 4.5))
plt.contour(xs, ys, zs, levels=np.logspace(-1, 1.3, 12), cmap="Greys", alpha=0.5)
for beta in (0.0, 0.5, 0.9):
    p = descend(beta)
    f_end = 0.5 * (p[-1, 0] ** 2 + 25 * p[-1, 1] ** 2)
    plt.plot(p[:, 0], p[:, 1], marker=".", ms=3, label=f"momentum {beta} (final loss {f_end:.3g})")
plt.xlabel("$w_1$"); plt.ylabel("$w_2$"); plt.title("Effect of momentum on an ill-conditioned quadratic (60 steps, lr=0.03)")
plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(OUT_DIR / "momentum_toy.png", dpi=150); plt.show()

## 4. Save results

In [ ]:
allres = dict(results)
allres.update({k: v for k, v in sweep.items() if all(v is not x for x in allres.values())})
summary = {k: {"best_val_acc": float(max(v["hist"]["val_acc"])),
               "epochs_to_90pct_val": epochs_to(v["hist"], 0.90),
               "test_acc": float(v["test_acc"]),
               "time_per_epoch_s": float(np.mean(v["hist"]["epoch_time"])),
               "history": dict(v["hist"])} for k, v in allres.items()}
with open(OUT_DIR / "optimizer_study.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved outputs/optimizer_study.json with", len(summary), "runs")

## 5. How to write the Part 3 answer (with your numbers)
1. **State the choice:** Adam, lr = 1e-3, plus the four reasons from Section 2.
2. **Quote the table** from Section 2: best validation accuracy, epochs to reach 90%, validation loss at epoch 5, test accuracy for Adam vs SGD vs SGD + momentum 0.9. Say which converged fastest and whether the final accuracies are close (a small gap means the *speed* of convergence, not the final accuracy, is the main difference).
3. **Explain the gap using the mechanisms:** plain SGD at lr = 0.01 has a small effective step on this un-normalised network; momentum multiplies it by about 1/(1-β); Adam rescales per parameter.
4. **Momentum impact:** use the sweep table and curves: where more momentum helped, where 0.99 became unstable or noisy, and how the smaller learning rate with β = 0.99 recovered. Support with the toy figure.
5. **Conclude** with the choice used for Models A and B and why it is justified by *your* curves.